# F3a — DrawBench: sinh ảnh LiDAR và RS-LiDAR (SD 1.5, DDPM-100, **một seed mỗi lần chạy**)

**Mục đích.** Ở lần mô phỏng reviewer thứ 3, cả ba reviewer xếp yêu cầu này lên đầu: một bộ prompt thứ hai với σ cố định từ trước. σ của bài được chọn trên GenEval, nên ở đây ta chạy đúng cấu hình dòng SD 1.5 DDPM-100 của Table 1 trên **DrawBench** (200 prompt, Saharia et al. 2022), **không chỉnh tham số nào**: σ = 1, M = 4, K = 50, λ = 5000.

Mọi lựa chọn (bộ prompt, cấu hình, seed, chỉ số, cách báo cáo) đã được chốt **trước khi chạy** trong `experiment_outputs/results/F3_drawbench/F3_preregistration.json`.

**Cách chạy**
- Chọn **L4 + High-RAM**.
- Đặt `SEED` ở mục 4: lần đầu là 100, các lần sau là 50 và 150.
- **Runtime → Run all.** Mỗi seed mất khoảng 2.5–3 giờ trên L4.
- Muốn thử pipeline trước thì đặt `SMOKE_TEST = True`. Notebook sẽ chạy 2 prompt vào thư mục `_smoke` riêng, không lẫn với run thật.

**Các bước**
1. Phase 1: tạo 50 ứng viên cho mỗi prompt. Bị ngắt thì Run all lại là chạy tiếp.
2. Chấm lại đúng 50 ứng viên đó bằng R̄σ (`--reuse_latents_from`, nhiễu i.i.d.).
3. Phase 2 cho LiDAR và RS-LiDAR. Phase 2 dùng seed toàn cục như Table 1 nên **mỗi run phải chạy một mạch**. Nếu bị ngắt, notebook in thư mục cần xoá rồi dừng.
4. In bảng IR/CLIP/HPS của seed này.

**Sau khi xong một seed:** mở **F3b** (L4 là đủ) để chấm PickScore và xem bảng tổng hợp. VQAScore chấm sau bằng **F3c** trên runtime A100 (High-RAM), lúc nào cũng được; F3c tự gộp với các chỉ số đã có.

**Kết quả:** `MyDrive/RS-LiDAR/extra_experiments/F3_drawbench/`. Ảnh nằm trong `Target_samples/F3DB_*`.

## 1. Kiểm Tra Phần Cứng GPU & VRAM
Đảm bảo bạn đã chọn Runtime: **Runtime -> Change runtime type -> A100 (hoặc L4) GPU** và bật **High-RAM**.

In [ ]:
import torch, sys, os
print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("CUDA khả dụng:", torch.cuda.is_available())

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2)
    print(f"🎮 GPU: {gpu_name}")
    print(f"💾 VRAM: {vram_gb} GB")
    if vram_gb >= 22.0:
        print("✅ Tuyệt vời! GPU có đủ VRAM để kích hoạt Full-Batch VAE Decoding tốc độ cao.")
    else:
        print("⚠️ Chú ý: GPU dưới 22GB VRAM, code sẽ tự động dùng chunking để bảo vệ bộ nhớ.")
else:
    raise RuntimeError("❌ Không phát hiện GPU CUDA! Vui lòng vào Runtime -> Change runtime type -> Chọn GPU.")

!nvidia-smi

## 2. Gắn Kết Google Drive & Thiết Lập Lưu Trữ Tức Thì (Real-time Persistence)
Toàn bộ kết quả sẽ được ghi trực tiếp vào `My Drive/RS-LiDAR/` thông qua symlink. Sau mỗi prompt hoàn thành, file `.png`, `latent.pt` và `results.json` được lưu vĩnh viễn trên Drive.

In [ ]:
import os, shutil, subprocess
from google.colab import drive

# 1. Gắn kết Google Drive an toàn
drive.mount('/content/drive')

base_drive = "/content/drive/My Drive" if os.path.exists("/content/drive/My Drive") else "/content/drive/MyDrive"
assert os.path.exists(base_drive), f"❌ Không tìm thấy thư mục Google Drive tại {base_drive}! Vui lòng gắn kết lại Drive."

DRIVE_DIR = f"{base_drive}/RS-LiDAR"
os.makedirs(f"{DRIVE_DIR}/Lookahead_samples", exist_ok=True)
os.makedirs(f"{DRIVE_DIR}/Target_samples", exist_ok=True)

# 2. Clone hoặc kéo cập nhật repo RS-LiDAR tự động
REPO_DIR = "/content/RS-LiDAR"

if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", "https://github.com/leekwanreal/RS-LiDAR.git", REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "fetch", "origin", "main"], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "reset", "--hard", "origin/main"], check=True)

WORKDIR = REPO_DIR if os.path.exists(f"{REPO_DIR}/lookahead_sampling.py") else f"{REPO_DIR}/Diffusion-LiDAR-Sampling"
%cd {WORKDIR}

# 3. Tạo Symbolic Links an toàn (Tuyệt đối không xóa dữ liệu cũ, tự động migrate nếu có file cục bộ)
for folder_name in ["Lookahead_samples", "Target_samples"]:
    link_path = os.path.join(WORKDIR, folder_name)
    drive_target = os.path.join(DRIVE_DIR, folder_name)
    os.makedirs(drive_target, exist_ok=True)

    if os.path.islink(link_path):
        if not os.path.exists(link_path):
            os.remove(link_path)
            os.symlink(drive_target, link_path)
    elif os.path.exists(link_path):
        print(f"📦 Phát hiện thư mục cục bộ {folder_name}, đang đồng bộ an toàn sang Google Drive...")
        for item in os.listdir(link_path):
            src_item = os.path.join(link_path, item)
            dst_item = os.path.join(drive_target, item)
            if not os.path.exists(dst_item):
                if os.path.isdir(src_item):
                    shutil.copytree(src_item, dst_item)
                else:
                    shutil.copy2(src_item, dst_item)
        shutil.rmtree(link_path)
        os.symlink(drive_target, link_path)
    else:
        os.symlink(drive_target, link_path)

print("✅ Hệ thống Symlink đã sẵn sàng 100%! Toàn bộ dữ liệu được bảo lưu vĩnh viễn trên Drive:")
print("  📁", DRIVE_DIR)

## 3. Cài Đặt Môi Trường Chuẩn Xác (100% Trơn Tru & Không Xung Đột)
Cài đặt bộ thư viện đồng bộ, tắt backend TensorFlow để chống xung đột Protobuf, và tải vocab HPSv2.

In [ ]:
import os, urllib.request

# 1. Tắt backend TensorFlow để loại bỏ xung đột Protobuf runtime_version
os.environ["USE_TF"] = "0"
os.environ["USE_TORCH"] = "1"

# 2. Nâng cấp protobuf và cài đặt các thư viện tương thích PyTorch 2.x
!pip install -q --upgrade protobuf open_clip_torch
!pip install -q transformers==4.38.2 diffusers==0.31.0 accelerate==1.2.1 safetensors huggingface-hub einops ftfy timm peft
!pip install -q git+https://github.com/openai/CLIP.git
!pip install -q image-reward || pip install -q --no-deps git+https://github.com/THUDM/ImageReward.git
!pip install -q hpsv2 matplotlib tqdm scipy seaborn pandas tabulate

# 3. Đảm bảo file từ điển OpenCLIP cho HPSv2
import hpsv2
hpsv2_vocab = os.path.join(os.path.dirname(hpsv2.__file__), "src", "open_clip", "bpe_simple_vocab_16e6.txt.gz")
os.makedirs(os.path.dirname(hpsv2_vocab), exist_ok=True)
if not os.path.exists(hpsv2_vocab):
    print("📥 Đang tải file BPE vocab cho HPSv2...")
    urllib.request.urlretrieve("https://github.com/openai/CLIP/raw/main/clip/bpe_simple_vocab_16e6.txt.gz", hpsv2_vocab)

print("✅ Cài đặt môi trường hoàn tất 100%!")

In [ ]:
# Kiểm tra môi trường sau khi cài: in phiên bản và dừng ngay nếu thiếu thư viện chấm điểm
# (thiếu hpsv2/clip thì code reward giờ báo lỗi thay vì chấm 0; dừng ở đây đỡ tốn GPU hơn).
import importlib, sys
if WORKDIR not in sys.path:
    sys.path.insert(0, WORKDIR)
from rslidar_utils import setup_runtime
setup_runtime()  # cùng các shim tương thích mà script của repo áp dụng trước khi import ImageReward
import torch, numpy, transformers, diffusers
print("python", sys.version.split()[0], "| torch", torch.__version__, "| numpy", numpy.__version__,
      "| transformers", transformers.__version__, "| diffusers", diffusers.__version__)
for mod in ("hpsv2", "clip", "open_clip", "ImageReward", "timm", "google.protobuf"):
    try:
        m = importlib.import_module(mod)
        print(f"  ✅ {mod} {getattr(m, '__version__', '')}")
    except Exception as e:
        raise RuntimeError(f"❌ Không import được {mod}: {e}. Chạy lại cell cài đặt (hoặc Runtime -> Restart session rồi chạy lại).")

## 4. Cấu hình (chỉ đổi `SEED`) và bộ prompt DrawBench

In [ ]:
import os, sys, glob, json, shutil, subprocess, hashlib, csv, io, urllib.request
import numpy as np, pandas as pd
os.environ["USE_TF"] = "0"
os.environ["USE_TORCH"] = "1"
%cd {WORKDIR}
if WORKDIR not in sys.path:
    sys.path.insert(0, WORKDIR)

# ============================== CHỈ ĐỔI HAI DÒNG NÀY ==============================
SEED = 100           # lần đầu 100, sau đó 50 rồi 150 (mỗi lần Run all một seed)
SMOKE_TEST = False   # True: chạy thử 2 prompt đầu (thư mục _smoke riêng) để kiểm tra pipeline
# ==================================================================================

EXP_NAME = "F3_drawbench"
EXP_DRIVE = f"{DRIVE_DIR}/extra_experiments/{EXP_NAME}"
os.makedirs(EXP_DRIVE, exist_ok=True)

# ---- Cấu hình dòng SD 1.5 DDPM-100 của Table 1; không chỉnh gì cho DrawBench (đã chốt trước) ----
MODEL_NAME = "runwayml/stable-diffusion-v1-5"
MODEL_TAG = "SD15"
STEPS_P1, K = 5, 50
SEED_P1 = SEED_P2 = SEED
SCALE, STEPS_P2, ETA_P2 = 12.5, 100, 1.0
NUM_PARTICLES_P2, RESAMPLE_T_END = 4, 200
LMBDA = 5000
RS_SIGMA, RS_M = 1.0, 4          # σ và M của Table 1, nhiễu i.i.d. (không dùng --use_crn)

# ---- DrawBench: file trong repo; nếu bản clone chưa có thì tạo từ bảng chính thức, rồi kiểm checksum ----
PROMPT_REL = "prompt_files/drawbench.jsonl"
PROMPT_SHA256 = "c47f23e6bb4ae883c0527cce1fb3052fc5cd8f34df9da9a7845c33996f041886"
DRAWBENCH_CSV = ("https://docs.google.com/spreadsheets/d/1y7nAbmR4FREi6npB1u-Bo3GFdwdOPYJc617rBOxIRHY"
                 "/gviz/tq?tqx=out:csv")   # bảng mà trang Imagen (imagen.research.google) trỏ tới
PROMPT_PATH = f"{WORKDIR}/{PROMPT_REL}"
if not os.path.exists(PROMPT_PATH):
    raw = urllib.request.urlopen(DRAWBENCH_CSV, timeout=60).read().decode("utf-8")
    rows = list(csv.DictReader(io.StringIO(raw, newline="")))
    lines = [json.dumps({"prompt": r["Prompts"], "category": r["Category"].strip(), "original_idx": i},
                        ensure_ascii=False) for i, r in enumerate(rows)]
    with open(PROMPT_PATH, "wb") as f:
        f.write(("\n".join(lines) + "\n").encode("utf-8"))
    print(f"📥 Đã tạo {PROMPT_REL} từ bảng DrawBench chính thức.")
_sha = hashlib.sha256(open(PROMPT_PATH, "rb").read()).hexdigest()
assert _sha == PROMPT_SHA256, (f"{PROMPT_REL} khác bản đã chốt ({_sha}). Dừng: không chạy với bộ prompt khác "
                               "bản trong F3_preregistration.json.")
META = [json.loads(l) for l in open(PROMPT_PATH, encoding="utf-8") if l.strip()]
assert len(META) == 200, len(META)
if not os.path.exists(f"{EXP_DRIVE}/drawbench.jsonl"):
    shutil.copyfile(PROMPT_PATH, f"{EXP_DRIVE}/drawbench.jsonl")   # F3b/F3c đọc nhóm prompt từ bản này

if SMOKE_TEST:
    PROMPT_INDICES_ARG, SUF, PROMPT_IDS = "0-1", "_smoke", [0, 1]
else:
    PROMPT_INDICES_ARG, SUF, PROMPT_IDS = None, "", list(range(len(META)))
MAX_PROMPT = len(META)

RAW_NAME = f"F3DB_Lookahead_{MODEL_TAG}_DPM{STEPS_P1}_n{K}_seed{SEED}{SUF}"
RS_NAME = f"F3DB_RSLookahead_{MODEL_TAG}_DPM{STEPS_P1}_n{K}_sig{RS_SIGMA}_M{RS_M}_seed{SEED}{SUF}"
LIDAR_RUN = f"F3DB_LiDAR_{MODEL_TAG}_n{K}_DDPM{STEPS_P2}_lmbda{LMBDA}_seed{SEED}{SUF}"
RS_RUN = f"F3DB_RSLiDAR_{MODEL_TAG}_n{K}_sig{RS_SIGMA}_M{RS_M}_DDPM{STEPS_P2}_lmbda{LMBDA}_seed{SEED}{SUF}"

json.dump({"seed": SEED, "smoke_test": SMOKE_TEST, "prompt_file": PROMPT_REL, "prompt_sha256": _sha,
           "n_prompts": len(PROMPT_IDS), "model": MODEL_NAME, "K": K, "lookahead_steps": STEPS_P1,
           "target": f"DDPM{STEPS_P2}", "eta": ETA_P2, "scale": SCALE, "lmbda": LMBDA, "sigma": RS_SIGMA,
           "M": RS_M, "noise_mode": "independent_iid", "num_images_per_prompt": NUM_PARTICLES_P2,
           "runs": {"lookahead": RAW_NAME, "rs_lookahead": RS_NAME, "LiDAR": LIDAR_RUN, "RS-LiDAR": RS_RUN}},
          open(f"{EXP_DRIVE}/F3_run_config_seed{SEED}{SUF}.json", "w"), indent=2)
print(f"F3 DrawBench | seed {SEED} | {len(PROMPT_IDS)} prompt{' (SMOKE TEST)' if SMOKE_TEST else ''}")
print("  LiDAR   :", LIDAR_RUN)
print("  RS-LiDAR:", RS_RUN)

In [ ]:
import torch
from tqdm.auto import tqdm


def n_have(d, ids=None):
    """Số prompt trong PROMPT_IDS đã có results.json trong thư mục d."""
    ids = PROMPT_IDS if ids is None else ids
    if not d or not os.path.isdir(d):
        return 0
    return sum(os.path.exists(f"{d}/{i:05d}/results.json") for i in ids)


def stream(cmd):
    print("$ " + " \\\n    ".join(cmd))
    proc = subprocess.Popen(cmd, cwd=WORKDIR, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
        print(line, end="")
    if proc.wait() != 0:
        raise RuntimeError(f"Lệnh thất bại (exit {proc.returncode}): {cmd[1]}")



def phase1(run_name, reuse_from=None, smoothing=False):
    """Phase 1 (lookahead_sampling.py) với đúng các cờ của notebook Table 1; bỏ qua nếu đã đủ.
    Bị ngắt thì chạy lại là tiếp tục (--resume); RS-LiDAR chấm lại đúng latent của pool LiDAR nên vẫn ghép cặp."""
    out = f"{WORKDIR}/Lookahead_samples/{run_name}"
    if n_have(out) >= len(PROMPT_IDS):
        print(f"✅ Phase 1 đã đủ {len(PROMPT_IDS)} prompt: {run_name}")
        return out
    cmd = ["python", "lookahead_sampling.py", f"--seed={SEED_P1}", f"--num_particles={K}",
           f"--num_inference_steps={STEPS_P1}", f"--model_name={MODEL_NAME}",
           "--guidance_reward_fn=ImageReward", "--metrics_to_compute=ImageReward",
           f"--prompt_path={PROMPT_REL}", f"--max_prompt={MAX_PROMPT}",
           f"--run_name={run_name}", "--resume"]
    if PROMPT_INDICES_ARG:
        cmd.append(f"--prompt_indices={PROMPT_INDICES_ARG}")
    if reuse_from:
        cmd.append(f"--reuse_latents_from={reuse_from}")
    if smoothing:
        cmd += ["--use_smoothing", f"--sigma={RS_SIGMA}", f"--num_mc_samples={RS_M}"]
    stream(cmd)
    assert n_have(out) >= len(PROMPT_IDS), f"Phase 1 chưa đủ prompt: {out}"
    return out


def phase2(lookahead_dir, run_name):
    """Phase 2 (LiDAR_sampling.py) với đúng các cờ của notebook Table 1 (seed toàn cục, không --per_prompt_seed)."""
    out = f"{WORKDIR}/Target_samples/{run_name}"
    have = n_have(out)
    if have >= len(PROMPT_IDS):
        print(f"✅ Phase 2 đã đủ {len(PROMPT_IDS)} prompt: {run_name}")
        return out
    if have > 0:
        raise RuntimeError(f"Run dở dang ({have}/{len(PROMPT_IDS)} prompt): {out}\n"
                           "Run này dùng seed toàn cục như Table 1 nên không chạy tiếp được. "
                           "Xoá thư mục trên (trên Drive: RS-LiDAR/Target_samples/...) rồi Run all lại.")
    cmd = ["python", "LiDAR_sampling.py", f"--seed={SEED_P2}", f"--model_name={MODEL_NAME}",
           f"--num_particles={NUM_PARTICLES_P2}", f"--num_inference_steps={STEPS_P2}", f"--eta={ETA_P2}",
           "--use_rag", f"--lookahead_path={lookahead_dir}", f"--top_k={K}", f"--scale={SCALE}",
           f"--lmbda={LMBDA}", f"--resample_t_end={RESAMPLE_T_END}",
           f"--prompt_path={PROMPT_REL}", f"--max_prompt={MAX_PROMPT}",
           "--guidance_reward_fn=ImageReward",
           "--metrics_to_compute=ImageReward#Clip-Score#Clip-Diversity#HumanPreference#AS",
           "--save_individual_images", f"--run_name={run_name}", "--resume"]
    if PROMPT_INDICES_ARG:
        cmd.append(f"--prompt_indices={PROMPT_INDICES_ARG}")
    stream(cmd)
    assert n_have(out) >= len(PROMPT_IDS), f"Phase 2 chưa đủ prompt: {out}"
    return out


def pools_paired(a, b, n_check=3):
    """True if pools a and b hold the same latents (up to order) for the first n_check prompts."""
    for i in PROMPT_IDS[:n_check]:
        if not (os.path.exists(f"{a}/{i:05d}/samples/latent.pt") and os.path.exists(f"{b}/{i:05d}/samples/latent.pt")):
            return False
        La, _ = load_pool(a, i)
        Lb, _ = load_pool(b, i)
        if La.shape != Lb.shape:
            return False
        A, Bm = La.flatten(1).float(), Lb.flatten(1).float()
        d = torch.cdist(Bm, A, compute_mode="donot_use_mm_for_euclid_dist").min(1).values
        if (d > 1e-3 * A.norm(dim=1).mean()).any():
            return False
    return True


def load_pool(d, i, key="ImageReward"):
    """Latents (K, 4, 64, 64) và reward (K,) của prompt i trong thư mục lookahead d."""
    lat = torch.load(f"{d}/{i:05d}/samples/latent.pt", map_location="cpu")
    lat = torch.stack(list(lat)) if isinstance(lat, (list, tuple)) else lat
    res = json.load(open(f"{d}/{i:05d}/results.json"))
    return lat, np.asarray(res[key]["result"], dtype=np.float64)


def rs_pool_ok(d, n_check=3):
    """Pool RS chỉ dùng được nếu được chấm đúng cấu hình bài: σ, M, nhiễu i.i.d., β = 0, làm trơn trên ảnh.

    Tên thư mục không ghi cách lấy nhiễu, nên phải đọc results.json["smoothing"]. Metadata có ba đời
    (rslidar_utils.read_smoothing_config): trước 27/10/2026 chỉ có use_crn (nhiễu i.i.d. hoặc CRN, không có
    entropic risk); 27/10 - 9/11 mặc định antithetic + β = 0.5; từ 9/11 mặc định i.i.d., β = 0.
    """
    from rslidar_utils import read_smoothing_config
    want = {"sigma": float(RS_SIGMA), "num_mc_samples": int(RS_M), "noise_mode": "independent_iid",
            "beta_risk": 0.0, "domain": "image"}
    legacy = False
    for i in PROMPT_IDS[:n_check]:
        f = f"{d}/{i:05d}/results.json"
        if not os.path.exists(f):
            return False
        sm = read_smoothing_config(json.load(open(f)))
        if not sm:
            print(f"⚠️ {os.path.basename(d)}: results.json không ghi cấu hình smoothing -> không dùng pool này.")
            return False
        bad = {k: (sm[k], v) for k, v in want.items()
               if (sm[k] != v if isinstance(v, str) else abs(float(sm[k]) - v) > 1e-9)}
        if bad:
            print(f"⚠️ {os.path.basename(d)}: cấu hình smoothing khác bài {bad} (có, cần) -> không dùng pool này.")
            return False
        legacy = legacy or sm["legacy_metadata"]
    if legacy:
        print(f"ℹ️ {os.path.basename(d)}: metadata đời cũ (trước 27/10/2026, chỉ có use_crn = False) "
              "-> nhiễu i.i.d., β = 0, đúng cấu hình bài.")
    return True


print("✅ Hàm tiện ích sẵn sàng.")

## 5. Phase 1: 50 ứng viên mỗi prompt, rồi chấm lại đúng các ứng viên đó bằng R̄σ (σ = 1, M = 4)

In [ ]:
RAW_DIR = phase1(RAW_NAME)
RS_DIR = phase1(RS_NAME, reuse_from=RAW_DIR, smoothing=True)
assert pools_paired(RAW_DIR, RS_DIR), "Pool RS không dùng chung latent với pool LiDAR -> dừng (so sánh sẽ không ghép cặp)."
assert rs_pool_ok(RS_DIR), "Pool RS không đúng cấu hình bài (σ = 1, M = 4, i.i.d., β = 0) -> dừng."
print("✅ Hai pool dùng chung 50 ứng viên; pool RS chấm bằng R̄σ đúng cấu hình bài.")

## 6. Phase 2: LiDAR và RS-LiDAR (seed toàn cục như Table 1, mỗi run chạy một mạch)

In [ ]:
RUNS = {}
print("=" * 90, "\n▶ LiDAR")
RUNS["LiDAR"] = phase2(RAW_DIR, LIDAR_RUN)
print("=" * 90, "\n▶ RS-LiDAR")
RUNS["RS-LiDAR"] = phase2(RS_DIR, RS_RUN)
print(RUNS)

## 7. Bảng nhanh của seed này (IR, CLIP, HPS)

In [ ]:
# Bảng nhanh cho seed này (IR, CLIP, HPS do Phase 2 ghi vào results.json). PickScore: F3b. VQAScore: F3c (A100).
METRICS = {"ImageReward": "IR", "Clip-Score": "CLIP", "HumanPreference": "HPS"}


def per_prompt(target_dir):
    out = {m: {} for m in METRICS}
    for i in PROMPT_IDS:
        r = json.load(open(f"{target_dir}/{i:05d}/results.json"))
        for m in METRICS:
            out[m][i] = float(r[m]["mean"])
    return out


P = {lab: per_prompt(d) for lab, d in RUNS.items()}
rows = []
for lab in RUNS:
    rows.append({"run": lab, **{s: np.mean([P[lab][m][i] for i in PROMPT_IDS]) for m, s in METRICS.items()}})
delta = {"run": "Δ (RS-LiDAR − LiDAR)"}
win = {"run": "RS-LiDAR thắng (% prompt)"}
for m, s in METRICS.items():
    a = np.array([P["RS-LiDAR"][m][i] for i in PROMPT_IDS])
    b = np.array([P["LiDAR"][m][i] for i in PROMPT_IDS])
    delta[s] = float((a - b).mean())
    win[s] = float(100 * ((a > b).mean() + 0.5 * (a == b).mean()))
df = pd.DataFrame(rows + [delta, win]).round(4)
display(df)
json.dump({"seed": SEED, "smoke_test": SMOKE_TEST, "n_prompts": len(PROMPT_IDS), "runs": RUNS,
           "table": df.to_dict(orient="records")},
          open(f"{EXP_DRIVE}/F3_gen_seed{SEED}{SUF}.json", "w"), indent=2)
print(f"Đã lưu {EXP_DRIVE}/F3_gen_seed{SEED}{SUF}.json. Tiếp theo: F3b (PickScore), sau đó F3c trên A100 (VQAScore).")

## Tự ngắt máy khi chạy xong
Cell cuối tự ngắt runtime Colab sau khi Run all xong (tiết kiệm compute unit). Kết quả đã lưu trên Drive.

In [ ]:
# Tự ngắt runtime khi Run all chạy xong, để không tốn thêm compute unit.
# Kết quả đã nằm trên Drive; đẩy hết dữ liệu đang chờ ghi lên Drive trước khi ngắt.
# Muốn giữ máy để xem/chạy tiếp thì đặt AUTO_DISCONNECT = False trước khi Run all.
AUTO_DISCONNECT = True
if AUTO_DISCONNECT:
    try:
        from google.colab import drive, runtime
        drive.flush_and_unmount()
        print("🛑 Đã chạy xong và lưu lên Drive. Tự ngắt runtime Colab.")
        runtime.unassign()
    except Exception as e:
        print("⚠️ Không tự ngắt được runtime, hãy tự ngắt: Runtime → Disconnect and delete runtime.", e)